<a href="https://colab.research.google.com/github/DanielRegaladoUMiami/sql-agent-llmops/blob/main/training/notebooks/train_chart_reasoner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Chart Reasoner · Fine-tuning Notebook

Fine-tune **Phi-3 Mini 3.8B** to map `(natural-language question + SQL result schema) → storytelling chart spec` using QLoRA with Unsloth.

- **Dataset**: [`DanielRegaladoCardoso/chart-reasoning-mix-v1`](https://huggingface.co/datasets/DanielRegaladoCardoso/chart-reasoning-mix-v1) — ~75 k rows (nvBench real + OpenAI storytelling synth)
- **Base model**: [`unsloth/Phi-3-mini-4k-instruct-bnb-4bit`](https://huggingface.co/unsloth/Phi-3-mini-4k-instruct-bnb-4bit)
- **Hardware**: Colab **T4 free tier** ✅
- **Expected time**: ~2–3 h for 1 epoch

> [`https://github.com/DanielRegaladoUMiami/sql-agent-llmops`](https://github.com/DanielRegaladoUMiami/sql-agent-llmops)

### Storytelling principles baked into the output schema

Synth rows include full chart spec: `chart_type`, `encoding`, **insight-driven title**, `subtitle`, `annotations`, `sort`, `color_strategy`, `highlight_value`, `axis_format`, `rationale` — distilled from Tufte / Knaflic / Few.


## 1 · Check GPU

In [1]:
!nvidia-smi

Sat Apr 18 12:23:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2 · Install

In [2]:
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets xformers


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.5 MB/s et

## 3 · HuggingFace login

In [3]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 4 · Load Phi-3 Mini in 4-bit

Unsloth's Phi-3 support has some version churn — this cell tries the
pre-quantized Unsloth repo first and falls back to Microsoft's official
Phi-3-mini-4k-instruct (with bitsandbytes 4-bit on the fly) if needed.

In [4]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

PRIMARY  = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
FALLBACK = "microsoft/Phi-3-mini-4k-instruct"

def load_with_fallback(primary, fallback, max_seq_length=MAX_SEQ_LEN):
    try:
        print(f"Trying primary: {primary}")
        return FastLanguageModel.from_pretrained(
            model_name=primary, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )
    except Exception as e:
        print(f"⚠️  Primary failed ({type(e).__name__}). Falling back to {fallback}.")
        return FastLanguageModel.from_pretrained(
            model_name=fallback, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )

model, tokenizer = load_with_fallback(PRIMARY, FALLBACK)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Trying primary: unsloth/Phi-3-mini-4k-instruct-bnb-4bit
==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

## 5 · Attach LoRA

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!
Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.6 patched 32 layers with 0 QKV layers, 32 O layers and 0 MLP layers.


## 6 · Load dataset

In [6]:
from datasets import load_dataset
import json

ds = load_dataset("DanielRegaladoCardoso/chart-reasoning-mix-v1")
print(ds)
print()
print("Sample instruction:", ds["train"][0]["instruction"])
print("Sample chart_spec keys:", list(ds["train"][0]["chart_spec"].keys())
      if isinstance(ds["train"][0]["chart_spec"], dict) else "(stored as JSON string)")


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/220k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/224k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33408 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/879 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/880 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'instruction', 'data_profile', 'chart_spec', 'source', 'difficulty'],
        num_rows: 33408
    })
    validation: Dataset({
        features: ['id', 'instruction', 'data_profile', 'chart_spec', 'source', 'difficulty'],
        num_rows: 879
    })
    test: Dataset({
        features: ['id', 'instruction', 'data_profile', 'chart_spec', 'source', 'difficulty'],
        num_rows: 880
    })
})

Sample instruction: For those employees whose salary is in the range of 8000 and 12000 and commission is not null or department number does not equal to 40, show me about the distribution of  job_id and the average of manager_id , and group by attribute job_id in a bar chart.
Sample chart_spec keys: (stored as JSON string)


## 7 · Format as chat messages

We teach the model to output the **full storytelling JSON spec** — not just
`chart_type` — so the resulting model produces insight-driven titles,
annotations, sorting rules, and rationales at inference time.

In [7]:
SYSTEM_PROMPT = (
    "You are an elite data-visualization expert. Given a natural-language "
    "question and the schema of a SQL result set, produce the ideal chart "
    "specification as JSON with: chart_type, encoding (x/y/color/size/facet), "
    "insight-driven title, subtitle, annotations, sort, color_strategy, "
    "highlight_value, axis_format, rationale. Respond with JSON only."
)

def _parse(v):
    """Parse a JSON-string field (how this dataset stores nested dicts)."""
    if isinstance(v, str):
        try: return json.loads(v)
        except: return {}
    return v or {}

def to_chat(row):
    profile = _parse(row["data_profile"])
    cols = profile.get("columns", [])
    user_content = (
        f"Question: {row['instruction']}\n\n"
        f"SQL result columns: {json.dumps(cols)}"
    )
    spec = row["chart_spec"] if isinstance(row["chart_spec"], str) else json.dumps(row["chart_spec"])
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": spec},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_ds = ds["train"].map(to_chat, remove_columns=ds["train"].column_names)
eval_ds  = ds["validation"].map(to_chat, remove_columns=ds["validation"].column_names)
print(train_ds[0]["text"][:1000])


Map:   0%|          | 0/33408 [00:00<?, ? examples/s]

Map:   0%|          | 0/879 [00:00<?, ? examples/s]

<|system|>
You are an elite data-visualization expert. Given a natural-language question and the schema of a SQL result set, produce the ideal chart specification as JSON with: chart_type, encoding (x/y/color/size/facet), insight-driven title, subtitle, annotations, sort, color_strategy, highlight_value, axis_format, rationale. Respond with JSON only.<|end|>
<|user|>
Question: For those employees whose salary is in the range of 8000 and 12000 and commission is not null or department number does not equal to 40, show me about the distribution of  job_id and the average of manager_id , and group by attribute job_id in a bar chart.

SQL result columns: [{"name": "JOB_ID", "type": "string"}, {"name": "AVG(MANAGER_ID)", "type": "number"}]<|end|>
<|assistant|>
{"chart_type": "bar", "encoding": {"x": "JOB_ID", "y": "AVG(MANAGER_ID)", "color": null, "size": null, "facet": null}, "title": "For those employees whose salary is in the range of 8000 and 12000 and commission is not null or departmen

## 7.5 · Filter rows that exceed MAX_SEQ_LEN

In [8]:
def fits(row):
    return len(tokenizer.encode(row["text"], add_special_tokens=False)) <= MAX_SEQ_LEN

_bt, _be = len(train_ds), len(eval_ds)
train_ds = train_ds.filter(fits, num_proc=2)
eval_ds  = eval_ds.filter(fits,  num_proc=2)
print(f"Train: kept {len(train_ds):,} / {_bt:,} ({100*len(train_ds)/max(1,_bt):.1f}%)")
print(f"Eval : kept {len(eval_ds):,} / {_be:,} ({100*len(eval_ds)/max(1,_be):.1f}%)")


Filter (num_proc=2):   0%|          | 0/33408 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/879 [00:00<?, ? examples/s]

Train: kept 33,408 / 33,408 (100.0%)
Eval : kept 879 / 879 (100.0%)


## 8 · Trainer config

In [9]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = "chart_reasoner_adapter",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.03,
    num_train_epochs            = 1,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    save_steps                  = 500,
    save_total_limit            = 2,
    optim                       = "adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    report_to                   = "none",
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds.select(range(min(500, len(eval_ds)))),
    args           = training_args,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/33408 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## 9 · Train

In [10]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 33,408 | Num Epochs = 1 | Total steps = 4,176
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,829,992,448 (0.23% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.955841
50,1.351950
75,0.487338
100,0.353633
125,0.310952
150,0.270032
175,0.276900
200,0.262512
225,0.243405
250,0.217627


TrainOutput(global_step=4176, training_loss=0.1766034843064702, metrics={'train_runtime': 6825.5648, 'train_samples_per_second': 4.895, 'train_steps_per_second': 0.612, 'total_flos': 2.346383338495488e+17, 'train_loss': 0.1766034843064702, 'epoch': 1.0})

## 10 · Inference test

In [11]:
FastLanguageModel.for_inference(model)

sample = ds["test"][0]
profile_s = _parse(sample["data_profile"])
cols = profile_s.get("columns", [])
user_content = f"Question: {sample['instruction']}\n\nSQL result columns: {json.dumps(cols)}"
print("Gold chart_spec:", _parse(sample["chart_spec"]))
print()

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids, max_new_tokens=600, do_sample=False, temperature=0)
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=600) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Gold chart_spec: {'chart_type': 'bar', 'encoding': {'x': 'name', 'y': 'count(*)', 'color': None, 'size': None, 'facet': None}, 'title': 'How many films are there in each category? List the genre name and the count with a bar chart, and rank in descending by', 'subtitle': None, 'annotations': [], 'sort': {'by': 'count(*)', 'order': 'desc'}, 'color_strategy': 'highlight', 'highlight_value': None, 'axis_format': {'y_scale': 'linear', 'y_label': None, 'x_label': None}, 'rationale': 'Bar chart compares count(*) across categories of name.'}



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


{"chart_type": "bar", "encoding": {"x": "name", "y": "count(*)", "color": null, "size": null, "facet": null}, "title": "How many films are there in each category? List the genre name and the count with a bar chart, and rank in descending by", "subtitle": null, "annotations": [], "sort": {"by": "count(*)", "order": "desc"}, "color_strategy": "highlight", "highlight_value": null, "axis_format": {"y_scale": "linear", "y_label": null, "x_label": null}, "rationale": "Bar chart compares count(*) across categories of name."}


## 11 · Push adapter to HuggingFace

In [15]:
model.push_to_hub_merged(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-lora",
    tokenizer,
    save_method="lora",
    token="**",
)


No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   6%|5         |  152MB / 2.65GB            

No files have been modified since last commit. Skipping to prevent empty commit.

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:02<00:00, 31.06s/it]


Unsloth: Merge process complete. Saved to `/content/DanielRegaladoCardoso/chart-reasoner-phi3-mini-lora`


In [14]:
model.push_to_hub(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-adapter-only",
    token="**",
)
tokenizer.push_to_hub(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-adapter-only",
    token="**",
)

README.md:   0%|          | 0.00/585 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 35.7MB / 35.7MB            

Saved model to https://huggingface.co/DanielRegaladoCardoso/chart-reasoner-phi3-mini-adapter-only


## ✅ Done

Your adapter is at:
**https://huggingface.co/DanielRegaladoCardoso/chart-reasoner-phi3-mini-lora**
